# NeuroSeg V2 — complete comparative study
Private Kaggle notebook: enable GPU and Internet. Run the cell. Train U-Net and Attention U-Net on the same patient-separated split, choose by validation Dice, test the frozen champion once, optionally track/register with MLflow, and export real results. Keep the original test set out of tuning. If MLflow installation fails, set TRACK_MLFLOW=False to run the study without it; registration then remains pending.

In [ ]:
import subprocess, sys, shutil, json
from pathlib import Path
import importlib.util
import torch

# Default: two architecture candidates. Set TUNE=True for two additional LR runs.
EPOCHS = 15
TUNE = False
TRACK_MLFLOW = True
MULTI_GPU = False  # Optional; DataParallel can be slower for small batches.

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."
repo = Path("/kaggle/working/neuroseg_code")
if not (repo / ".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch","neuroseg-v2",
                    "https://github.com/Manz203/brain-tumor-segmentation.git",str(repo)],check=True)
else:
    subprocess.run(["git","-C",str(repo),"pull","--ff-only"],check=True)

required={"h5py":"h5py","sklearn":"scikit-learn","pandas":"pandas","PIL":"Pillow","matplotlib":"matplotlib"}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable,"-m","pip","install",*missing],check=True)
if TRACK_MLFLOW:
    subprocess.run([sys.executable,"-m","pip","install","-r",str(repo/"requirements-mlops.txt")],check=True)

out=Path("/kaggle/working/neuroseg_study")
cmd=[sys.executable,"-u","-m","neuroseg.study","--epochs",str(EPOCHS),"--out",str(out)]
if TUNE: cmd.append("--tune")
if TRACK_MLFLOW: cmd.append("--mlflow")
if MULTI_GPU: cmd.append("--multi-gpu")
subprocess.run(cmd,cwd=repo,check=True)

# Package genuine results and the best candidate of each architecture for the demo.
import pandas as pd
bundle=Path("/kaggle/working/neuroseg_submission")
bundle.mkdir(exist_ok=True)
for name in ["best_model.pth","selection.json","model_comparison.csv","test_metrics.json",
             "training_history.csv","training_curves.png","segmentation_examples.png","per_sample_metrics.csv",
             "dataset_manifest.csv","registry.json"]:
    path=out/name
    if path.is_file(): shutil.copy2(path,bundle/name)
table=pd.read_csv(out/"model_comparison.csv").sort_values("validation_dice",ascending=False)
for row in table.drop_duplicates("architecture").itertuples():
    dest=bundle/row.candidate
    dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2(out/row.candidate/"best_unet.pth",dest/"best_unet.pth")
    shutil.copy2(out/row.candidate/"training_history.csv",dest/"training_history.csv")

archive=shutil.make_archive("/kaggle/working/neuroseg_submission","zip",root_dir=bundle)
print("RESULTS ZIP:",archive)
print((out/"test_metrics.json").read_text())
